# placax: MaskPlace training on Colab

Clones `placax`, installs it with GPU support, and runs `scripts/run_maskplace.py` on a Colab GPU runtime (16GB on the free T4 tier vs. 4GB locally, so `--n_episodes=auto` should pick something higher than 1).

Before running: **Runtime > Change runtime type > T4 GPU** (or better, if you have Colab Pro).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# Clone the repo. If it's private, either paste a token below
# (https://github.com/settings/tokens, "repo" scope) or use
# Colab's "Files > mount GitHub" flow instead.
GITHUB_TOKEN = ""  # leave empty for a public repo

repo_url = (
    f"https://{GITHUB_TOKEN}@github.com/paulin-dev/placax.git"
    if GITHUB_TOKEN else
    "https://github.com/paulin-dev/placax.git"
)
!git clone $repo_url
%cd placax

In [ ]:
# Colab's runtime already ships a working CUDA-enabled JAX - installing placax's own
# .[cuda] extra on top of it registers a second, conflicting 'cuda' PJRT plugin (you'd
# see a "PJRT_Api already exists for device type cuda" error at import time). Just
# install placax itself plus the resnet extra and reuse Colab's preinstalled JAX.
!pip install -q -e .[resnet]

In [ ]:
import jax
print(jax.devices())  # should list a Gpu/Cuda device, not just Cpu

## (optional) checkpoint to Drive

Colab runtimes are ephemeral - if you want the checkpoint/training log to survive a
disconnect, point the run at a Drive-backed output dir instead of the repo's local one.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import pathlib
    drive_out = pathlib.Path("/content/drive/MyDrive/placax/adaptec1_output_maskplace")
    drive_out.mkdir(parents=True, exist_ok=True)
    local_out = pathlib.Path("benchmarks/adaptec1/output_maskplace")
    if local_out.is_symlink() or local_out.exists():
        import shutil
        if local_out.is_symlink():
            local_out.unlink()
        else:
            shutil.rmtree(local_out)
    local_out.symlink_to(drive_out)

In [ ]:
!python -m scripts.run_maskplace --n_episodes=auto --n_iterations=300